# Capítulo 6 – Modelo Final y Evaluación con Validación Cruzada
## Sistema de Bicicletas Compartidas – Dataset `hour_prepared.csv`

En este capítulo construiremos un **modelo final de regresión lineal múltiple** para la demanda horaria de bicicletas (`cnt`).

Usaremos el dataset preparado en el Capítulo 2 (`hour_prepared.csv`), y:

- Seleccionaremos un subconjunto de variables mediante **selección forward basada en AIC**.
- Ajustaremos el modelo final con todas las observaciones.
- Evaluaremos su desempeño usando **validación cruzada K-fold (K = 5)**.
- Analizaremos gráficos básicos de ajuste del modelo final.


## 1. Carga de librerías y datos

Cargamos el dataset ya preparado que contiene las variables numéricas y dummies listas para modelar.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11


In [ ]:
# Ruta relativa al dataset preparado
data_path = "../data/hour_prepared.csv"
df_model = pd.read_csv(data_path)
df_model.head()

Verificamos dimensiones y ausencia de valores faltantes.


In [ ]:
df_model.shape

In [ ]:
df_model.isna().sum().sum()

## 2. Definición de la variable objetivo y predictores

La variable objetivo es `cnt` y los predictores son todas las demás columnas del dataset.


In [ ]:
target_col = "cnt"
if target_col not in df_model.columns:
    raise ValueError("La columna 'cnt' no se encuentra en 'hour_prepared.csv'.")

y = df_model[target_col].copy()
X_full = df_model.drop(columns=[target_col]).copy()

X_full.shape, y.shape

## 3. Selección de variables mediante Forward Selection (AIC)

Implementamos un procedimiento de **selección forward** usando el criterio AIC:

1. Comenzamos sin predictores.
2. En cada paso, probamos agregar cada variable candidata restante.
3. Elegimos la que produce el menor AIC.
4. Si el AIC mejora respecto al modelo anterior, la añadimos al conjunto.
5. Repetimos hasta que añadir nuevas variables ya no mejore el AIC.


In [ ]:
def fit_ols_add_const(X, y):
    X_const = sm.add_constant(X, has_constant='add')
    model = sm.OLS(y, X_const).fit()
    return model

all_predictors = list(X_full.columns)
selected_vars = []
current_aic = None
history = []

while True:
    remaining = [v for v in all_predictors if v not in selected_vars]
    if not remaining:
        break
    trial_results = []
    for var in remaining:
        vars_trial = selected_vars + [var]
        model_trial = fit_ols_add_const(X_full[vars_trial], y)
        trial_results.append((model_trial.aic, var, model_trial))
    trial_results.sort(key=lambda x: x[0])
    best_aic, best_var, best_model = trial_results[0]
    if current_aic is None or best_aic < current_aic - 1e-6:
        selected_vars.append(best_var)
        current_aic = best_aic
        history.append((len(selected_vars), best_var, best_aic))
    else:
        break

selected_vars, current_aic

In [ ]:
history_df = pd.DataFrame(history, columns=["k", "variable_agregada", "AIC"])
history_df

El conjunto de variables seleccionadas por AIC será la base de nuestro **modelo final**.


In [ ]:
X_sel = X_full[selected_vars].copy()
X_sel.shape

## 4. Ajuste del modelo final con todas las observaciones

Ajustamos el modelo OLS usando **todas las observaciones** y solo las variables seleccionadas.


In [ ]:
X_sel_const = sm.add_constant(X_sel, has_constant='add')
model_final = sm.OLS(y, X_sel_const).fit()
model_final

Mostramos el resumen del modelo final:


In [ ]:
model_final.summary()

### 4.1. Tabla de coeficientes del modelo final

Construimos una tabla con coeficientes, errores estándar, estadísticos t, valores-p e intervalos de confianza al 95%.


In [ ]:
conf_int = model_final.conf_int(alpha=0.05)
conf_int.columns = ["IC 2.5%", "IC 97.5%"]

coef_table = pd.DataFrame({
    "coef": model_final.params,
    "std err": model_final.bse,
    "t": model_final.tvalues,
    "P>|t|": model_final.pvalues,
})
coef_table = pd.concat([coef_table, conf_int], axis=1)
coef_table.index.name = "Parámetro"
coef_table.round(4)

## 5. Validación cruzada K-fold (K = 5)

Para evaluar la capacidad de generalización del modelo final realizamos una validación cruzada K-fold con K = 5. En cada partición:

1. Ajustamos el modelo en el conjunto de entrenamiento.
2. Calculamos el RMSE en el conjunto de validación.
3. Promediamos los RMSE de las 5 particiones.


In [ ]:
def cv_rmse_ols(X, y, k=5, random_state=123):
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    rmses = []
    for train_idx, test_idx in kf.split(X):
        X_train_cv = X.iloc[train_idx]
        y_train_cv = y.iloc[train_idx]
        X_test_cv = X.iloc[test_idx]
        y_test_cv = y.iloc[test_idx]

        X_train_cv_const = sm.add_constant(X_train_cv, has_constant='add')
        X_test_cv_const = sm.add_constant(X_test_cv, has_constant='add')
        model_cv = sm.OLS(y_train_cv, X_train_cv_const).fit()
        y_pred_cv = model_cv.predict(X_test_cv_const)
        rmse_cv = np.sqrt(mean_squared_error(y_test_cv, y_pred_cv))
        rmses.append(rmse_cv)
    return np.mean(rmses), np.std(rmses)

mean_rmse_cv, std_rmse_cv = cv_rmse_ols(X_sel, y, k=5)
mean_rmse_cv, std_rmse_cv

También calculamos las métricas **in-sample** (usando todas las observaciones):


In [ ]:
y_pred_full = model_final.predict(X_sel_const)
rmse_full = np.sqrt(mean_squared_error(y, y_pred_full))
r2_full = r2_score(y, y_pred_full)

metrics_final = pd.DataFrame({
    "RMSE": [rmse_full, mean_rmse_cv],
    "R2": [r2_full, np.nan],
}, index=["Full sample", "CV (mean)"])
metrics_final.round(4)

## 6. Gráficos de ajuste del modelo final

Para complementar el análisis numérico, mostramos:

- Diagrama de dispersión `cnt observado` vs `cnt predicho`.
- Gráfico de residuales vs valores ajustados.


In [ ]:
# Observado vs predicho
fig, ax = plt.subplots()
ax.scatter(y, y_pred_full, alpha=0.3)
ax.set_xlabel("cnt observado")
ax.set_ylabel("cnt predicho")
ax.set_title("Modelo final: observado vs predicho")
plt.tight_layout()
plt.show()

In [ ]:
# Residuos vs valores ajustados
residuals_full = y - y_pred_full
fig, ax = plt.subplots()
ax.scatter(y_pred_full, residuals_full, alpha=0.3)
ax.axhline(0, linestyle='--')
ax.set_xlabel("Valores ajustados (modelo final)")
ax.set_ylabel("Residuos")
ax.set_title("Modelo final: residuos vs valores ajustados")
plt.tight_layout()
plt.show()

## 7. Conclusiones del modelo final

En este capítulo:

- Aplicamos **selección forward basada en AIC** para escoger un subconjunto de variables explicativas a partir de `hour_prepared.csv`.
- Ajustamos un **modelo final de regresión lineal múltiple** con todas las observaciones.
- Calculamos métricas de ajuste global (RMSE y R² en la muestra completa).
- Evaluamos la capacidad de generalización mediante **validación cruzada K-fold**.
- Visualizamos el desempeño del modelo final mediante gráficos de observado vs predicho y residuales vs ajustados.

En capítulos posteriores se podrían explorar extensiones como:

- Inclusión de términos polinómicos o interacciones.
- Modelos regularizados (Ridge, Lasso, ElasticNet).
- Modelos no lineales o basados en árboles de decisión.
